# Kiểm chứng số liệu — Atmospheric Remote Sensing (USTH)

Notebook tính lại **độc lập** mọi con số xuất hiện trong bài giảng, slide, quiz và công cụ tính của site, rồi so với `../data/verified.json` (do `build/verify_numbers.py` sinh). Công thức lấy từ đề cương *Satellite Remote Sensing of Atmospheric Composition*. Các mô hình đơn giản (TIR, O₂ A-band, mô phỏng averaging kernel) là minh họa tự dựng, không phải dữ liệu đo.

In [1]:
import json, math, numpy as np
from scipy.optimize import brentq
V = json.load(open('../data/verified.json'))
H,C,KB = 6.62607015e-34, 2.99792458e8, 1.380649e-23
def chk(name, a, b, rel=1e-6):
    ok = abs(a-b) <= rel*max(1,abs(b)); print(('OK  ' if ok else 'FAIL'), name, a, b); assert ok

## Ch.1 — Thời gian sống
τ = 1/(k[OH]); gộp song song 1/τ = Σ 1/τᵢ; plume τ = L/U; suy giảm e^(−t/τ).

In [2]:
k, oh = 2e-13, 1e6
tau = 1/(k*oh)
chk('CO lifetime days', tau/86400, V['co_oh']['tau_days'])
chk('parallel 2,3,6 d', 1/(1/2+1/3+1/6), V['par_example']['total'])
chk('plume tau h (L=50 km, U=5 m/s)', 50e3/5/3600, V['plume']['tau_h'])
# khớp hàm mũ trên dữ liệu tổng hợp
x = np.linspace(0,150e3,31); y = np.exp(-x/50e3); chk('fit e-fold', -1/np.polyfit(x,np.log(y),1)[0], 50e3, 1e-9)
chk('e^-1', math.exp(-1), V['decay']['e1'])
print('half-life/tau =', math.log(2))

OK   CO lifetime days 57.870370370370374 57.870370370370374
OK   parallel 2,3,6 d 1.0000000000000002 1.0
OK   plume tau h (L=50 km, U=5 m/s) 2.7777777777777777 2.7777777777777777
OK   fit e-fold 49999.999999999985 50000.0
OK   e^-1 0.36787944117144233 0.36787944117144233
half-life/tau = 0.6931471805599453


## Ch.2 — Beer–Lambert, Planck, Wien, Stefan–Boltzmann, Rayleigh

In [3]:
tv, sza = 0.5, 60
ts = tv/math.cos(math.radians(sza)); chk('slant tau', ts, V['beer']['tau_slant']); chk('T', math.exp(-ts), V['beer']['T'])
B = lambda lam,T: 2*H*C**2/lam**5/(math.exp(H*C/(lam*KB*T))-1)
for name,T in [('sun',5778),('earth',288)]:
    from scipy.optimize import minimize_scalar
    lmax = minimize_scalar(lambda l: -B(l,T), bounds=(0.1e-6,60e-6), method='bounded', options=dict(xatol=1e-12)).x*1e6
    print(name,'numeric lambda_max (um)', round(lmax,4),' Wien', round(2898/T,4)); assert abs(lmax-2898/T)/(2898/T) < 0.01
chk('Stefan 288 K', 5.670374419e-8*288**4, V['stefan']['M_earth'], 1e-9)
chk('Rayleigh 450 vs 650', (650/450)**4, V['rayleigh_450_650'])

OK   slant tau 0.9999999999999998 0.9999999999999998
OK   T 0.3678794411714424 0.3678794411714424
sun numeric lambda_max (um) 0.5015  Wien 0.5016
earth numeric lambda_max (um) 10.0617  Wien 10.0625
OK   Stefan 288 K 390.1051535418615 390.1051535418615
OK   Rayleigh 450 vs 650 4.353147386069196 4.353147386069196


## Ch.3 — AMF, VCD, Dobson Unit

In [4]:
amf = 1/math.cos(math.radians(60)) + 1/math.cos(math.radians(30))
chk('AMF', amf, V['amf']['amf']); chk('VCD', (8e15-3e15)/amf, V['amf']['vcd'], 1e-9)
print('VCD in DU if O3:', (8e15-3e15)/amf/2.687e16)
chk('300 DU', 300*2.687e16, V['DU']['example_300DU'], 1e-9)

OK   AMF 3.154700538379251 3.154700538379251
OK   VCD 1584936490538903.8 1584936490538903.8
VCD in DU if O3: 0.05898535506285463
OK   300 DU 8.061e+18 8.061e+18


## Ch.5 — Optimal estimation: averaging kernel và DOFS
Mô hình 12 mực: K Gauss rộng w, S_ε = σ_ε² I, S_a có tương quan exp(−|i−j|/L). A = (KᵀS_ε⁻¹K + S_a⁻¹)⁻¹KᵀS_ε⁻¹K; DOFS = tr(A) = tổng trị riêng.

In [5]:
def oe(N=12,w=3.0,se=0.1,sa=1.0,Lc=3.0):
    z=np.arange(N); K=np.exp(-0.5*((z[None,:]-z[:,None])/w)**2); Sa=sa**2*np.exp(-np.abs(z[:,None]-z[None,:])/Lc)
    Sei=np.eye(N)/se**2; M=np.einsum('ij,jk->ik',np.einsum('ij,jk->ik',K.T,Sei),K)+np.linalg.inv(Sa)
    G=np.einsum('ij,jk->ik',np.linalg.inv(M),np.einsum('ij,jk->ik',K.T,Sei)); A=np.einsum('ij,jk->ik',G,K); return A
for name,p in dict(ideal=(0.3,0.01),realistic=(3,0.1),noisy=(3,1.0)).items():
    A=oe(12,*p); chk('DOFS '+name, np.trace(A), V['oe'][name]['dofs'], 1e-9); chk('sum eig '+name, np.linalg.eigvals(A).real.sum(), np.trace(A), 1e-9)
    assert np.allclose(A, np.array(V['oe'][name]['A']), atol=1e-9)
print('A tinh lai khop toan bo ma tran 12x12 cho 3 truong hop')

OK   DOFS ideal 11.99645488442944 11.99645488442944
OK   sum eig ideal 11.99645488442944 11.99645488442944
OK   DOFS realistic 4.341169832865099 4.341169832865099
OK   sum eig realistic 4.3411698328650985 4.341169832865099
OK   DOFS noisy 2.89276948595352 2.89276948595352
OK   sum eig noisy 2.892769485953522 2.89276948595352
A tinh lai khop toan bo ma tran 12x12 cho 3 truong hop


## Ch.6 — Ångström

In [6]:
a=math.log(0.60/0.15)/math.log(870/440); chk('alpha', a, V['angstrom']['alpha']); chk('AOD550', 0.60*(550/440)**(-a), V['angstrom']['tau550'])
ls=np.array([440,500,675,870,1020.]); t=0.6*(ls/440)**(-a); chk('regression alpha', -np.polyfit(np.log(ls),np.log(t),1)[0], a, 1e-9)

OK   alpha 2.033529077117641 2.033529077117641
OK   AOD550 0.3811377107074469 0.3811377107074469
OK   regression alpha 2.0335290771176373 2.033529077117641


## Ch.7 — Tham số kích thước và hiệu suất suy giảm Q_ext (Lorenz–Mie tự cài, Bohren–Huffman)

In [7]:
xp=lambda r,l:2*math.pi*r/l
chk('x N2',xp(2e-4,.55),V['size']['N2']); chk('x sulfate',xp(.2,.55),V['size']['sulfate']); chk('x dust',xp(5,.55),V['size']['dust'])
def qext(m,x):
    nmax=int(x+4*x**(1/3)+2); mx=m*x; nmx=int(max(nmax,abs(mx))+16); D=np.zeros(nmx+1,dtype=complex)
    for n in range(nmx,0,-1): D[n-1]=n/mx-1/(D[n]+n/mx)
    psi0,psi1=math.cos(x),math.sin(x); chi0,chi1=-math.sin(x),math.cos(x); xi1=complex(psi1,-chi1); q=0
    for n in range(1,nmax+1):
        psi=(2*n-1)/x*psi1-psi0; chi=(2*n-1)/x*chi1-chi0; xi=complex(psi,-chi)
        a_=((D[n]/m+n/x)*psi-psi1)/((D[n]/m+n/x)*xi-xi1); b_=((D[n]*m+n/x)*psi-psi1)/((D[n]*m+n/x)*xi-xi1)
        q+=(2*n+1)*(a_.real+b_.real); psi0,psi1,chi0,chi1,xi1=psi1,psi,chi1,chi,xi
    return 2/x**2*q
for x in [0.1,0.5,1,2,5,10,20,50,100,200]: chk('Qext x=%g'%x, qext(1.33,x), V['mie_qext_m133'][str(x)], 1e-9)
m=1.33;xr=0.05;qr=8/3*xr**4*abs((m**2-1)/(m**2+2))**2; print('Rayleigh check x=0.05:', qext(m,xr), qr); assert abs(qext(m,xr)-qr)/qr<0.02

OK   x N2 0.002284794657156213 0.002284794657156213
OK   x sulfate 2.2847946571562128 2.2847946571562128
OK   x dust 57.119866428905326 57.119866428905326
OK   Qext x=0.1 1.109062536221262e-05 1.109062536221262e-05
OK   Qext x=0.5 0.006773139883838112 0.006773139883838112
OK   Qext x=1 0.09392400121407181 0.09392400121407181
OK   Qext x=2 0.7129483218556671 0.7129483218556671
OK   Qext x=5 3.5910329236295633 3.5910329236295633
OK   Qext x=10 2.2065487101846193 2.2065487101846193
OK   Qext x=20 2.1401071524274493 2.1401071524274493
OK   Qext x=50 1.9798862486569728 1.9798862486569728
OK   Qext x=100 2.1010699120374077 2.1010699120374077
OK   Qext x=200 2.0556193568075147 2.0556193568075147
Rayleigh check x=0.05: 6.935521560573177e-07 6.936801105674503e-07


## Ch.8 — Hồng ngoại nhiệt và O₂ A-band (mô hình đơn giản)

In [8]:
pl_inv=lambda lam,Bv: H*C/(lam*KB)/math.log(1+2*H*C**2/(lam**5*Bv))
def bt(Ts,z,tau,lam=10e-6,lap=6.5):
    Tl=Ts-lap*z; I=B(lam,Ts)*math.exp(-tau)+B(lam,Tl)*(1-math.exp(-tau)); return pl_inv(lam,I),Tl
b1,_=bt(300,1,.5); b5,_=bt(300,5,.5); chk('dBT 1 km',300-b1,V['tir']['z1']['dBT'],1e-9); chk('dBT 5 km',300-b5,V['tir']['z5']['dBT'],1e-9)
t_low=brentq(lambda t:(300-bt(300,1,t)[0])-(300-bt(300,5,.1)[0]),0.001,20); chk('degenerate tau (1 km)',t_low,V['tir_degenerate']['low_thick']['tau'],1e-8)
for z in (1,3,5,8): chk('O2 above %d km'%z, math.exp(-z/7), V['oxygen_above'][str(z)])
print('Lop 5 km cho dBT lon hon', round((300-b5)/(300-b1),2),'lan lop 1 km')

OK   dBT 1 km 2.508193864756265 2.508193864756265
OK   dBT 5 km 11.455202740120455 11.455202740120399
OK   degenerate tau (1 km) 0.5376165784494953 0.5376165784494724
OK   O2 above 1 km 0.8668778997501816 0.8668778997501816
OK   O2 above 3 km 0.6514390575310556 0.6514390575310556
OK   O2 above 5 km 0.4895416595569531 0.4895416595569531
OK   O2 above 8 km 0.31890655732397044 0.31890655732397044
Lop 5 km cho dBT lon hon 4.57 lan lop 1 km


**Kết luận:** mọi phép so sánh đều `OK` (nếu có `FAIL` thì ô tương ứng đã dừng bằng `assert`). File JS `site/_shared/calc.js` được kiểm riêng bằng `tests/test_calc.js` (455 phép so sánh với cùng `verified.json`).